In [ ]:
# imports
import torch
from models.bert import ContinuityBERT
from train import parse_args
import create_knowledge_graph as kg_utils
from data import utils

In [2]:
# Load the right model that we want to analyze
def create_bert_model(config):
    encoder_type = config["encoder_type"]
    use_kg = "kg" in config["model_type"]
    model = ContinuityBERT(
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        n_gnn_layers=config["n_gnn_layers"],
        hidden_dim=config["hidden_dim"],
        input_dim=utils.SENTENCE_ENCODER_DIM[encoder_type],
        use_kg=use_kg,
        kg_node_dim=kg_utils.KG_NODE_DIM,
        kg_edge_dim=kg_utils.KG_EDGE_DIM,
        dropout=config["dropout"],
        gnn_type=config["gnn_type"],
    )
    return model

config_bert_kg_gat = {
    "train_ratio": 0.5,
    "batch_size": 64,
    "n_continuity_errors": 1, #[1, 2
    "n_heads": 8,
    "n_layers": 3,
    "n_gnn_layers": 2,
    "hidden_dim": 20,
    "dropout": 0.2,
    "n_epochs": 100,
    "n_runs": 5,
    "lr": 1e-5,
    "pr_threshold": 0.3,
    "encoder_type": "all-MiniLM-L6-v2",
    "gnn_type": "gatv2", #["gatv2", "gcn"],
    "model_type": "bert_kg"
}

model = create_bert_model(config_bert_kg_gat)

initialized continuityBERT with 628261 parameters.


In [3]:
# Load saved weights into 
MODEL_WEIGHTS_PATH = "./results/bert_kg_gat/bert_kg_gat-1_error-params.pkl"
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH))

<All keys matched successfully>

In [4]:
# Load example data
import pickle as pkl
input_data = "./data/dataset/encoded/test/test_1_error.pkl"
with open(input_data, "rb") as f:
    dataset, _ = pkl.load(f)

xs, ys = [], []
kgs = []
docs = []
for i, (x, y, kg, doc) in enumerate(dataset):
    xs.append(x)
    ys.append(y)
    kgs.append(kg)
    docs.append(doc)

In [82]:
example_datapoint = 24
x = xs[example_datapoint]
y = ys[example_datapoint]
kg = kgs[example_datapoint]

# datapoint 5 == 1_error/test/synthetic_kaggle_2000_776_continuity7.txt
# datapoint 8 == .../synthetic_kaggle_2254_779_continuity4.txt
import os
osl = os.listdir
ospj = os.path.join
orig_docs_path = "data/dataset/1_error/test/"
def find_original_doc(kg, y, kgidx=0):
    node_labels = kg["node_labels"]
    if len(node_labels) <= kgidx: return [(None, None)]
    ds = [x for x in osl(orig_docs_path) if x.endswith(".txt")]
    matches = []
    for d in ds:
        with open(ospj(orig_docs_path, d)) as f:
            lines = f.readlines()
        txtmatch = node_labels[kgidx]
        max_idx = torch.argmax(y).item()
        #if d == "synthetic_kaggle_2701_902_continuity6.txt":
        #    print(max_idx, lines[0])
        #    print(str(max_idx) in lines[0])
        #    print(txtmatch in " ".join(lines))
        #    print(txtmatch)
        if not (str(max_idx) in lines[0] and txtmatch in " ".join(lines)):
            continue
        matches.append((d, lines))
    if not matches:
        print(f"WARNING: no match found for kgidx={kgidx}!")
        return find_original_doc(kg, y, kgidx=kgidx+1)
        matches = [(None, None)]
    # filter matches if more than 1
    #while len(matches) > 1:
    #    for 
    print(f"Success! Matching doc(s) found at kgidx={kgidx}")
    return matches#[0]
        
        

print(f"data_{example_datapoint} # sentences: {len(x)}")
orig_data_file, orig_data_lines = find_original_doc(kg, y)[0]
print(f"Using data file: {orig_data_file}")

data_24 # sentences: 126
Success! Matching doc(s) found at kgidx=1
Using data file: synthetic_kaggle_2701_902_continuity6.txt


In [71]:
y_hat = model.forward(x.reshape([1, len(x), -1]), [kg])[0]

# Get solutions
def top_idxs_ordered_desc(input: torch.Tensor):
    return [y[1] for y in sorted([(x, i) for i, x in enumerate(input)])[::-1]]

max_idx = torch.argmax(y)
max_idx_hat = torch.argmax(y_hat)
print("max index y:", max_idx)
ordered_desc_idxs = top_idxs_ordered_desc(y_hat)
print("ordered desc indices y:", ordered_desc_idxs)
print("len desc indices y:", len(ordered_desc_idxs))

matched = max_idx == max_idx_hat
print(f"Is correct?: {matched}")

max index y: tensor(18)
ordered desc indices y: [18, 57, 28, 47, 123, 48, 120, 2, 59, 38, 75, 96, 81, 17, 72, 88, 46, 51, 16, 66, 116, 19, 20, 93, 55, 114, 98, 97, 32, 122, 102, 54, 111, 41, 12, 24, 94, 44, 37, 39, 92, 112, 33, 43, 4, 1, 50, 62, 23, 99, 3, 63, 124, 108, 9, 82, 73, 125, 115, 61, 119, 103, 118, 8, 29, 101, 14, 10, 65, 78, 34, 70, 74, 90, 80, 25, 7, 11, 106, 22, 109, 31, 69, 83, 84, 76, 91, 79, 56, 113, 58, 36, 60, 117, 87, 95, 42, 49, 13, 71, 40, 121, 52, 77, 0, 68, 105, 6, 30, 5, 86, 27, 53, 45, 107, 110, 15, 35, 85, 89, 21, 64, 104, 67, 26, 100]
len desc indices y: 126
Is correct?: True
